In [1]:
from random import random
from functools import reduce
from collections import namedtuple
from queue import PriorityQueue, SimpleQueue, LifoQueue
import numpy as np

In [75]:
PROBLEM_SIZE = 5  # numero di elementi in un set
NUM_SETS = 10
#Crea un array di array dove un elemento ha il 30% di probabilità di essere true. Un set indica quale item c'è all'interno del set e quale no
SETS = tuple(np.array([random() < .3 for _ in range(PROBLEM_SIZE)]) for _ in range(NUM_SETS))
State = namedtuple('State', ['taken', 'not_taken'])

# uno stato è fatto da: i set di item che prendo e i set che non prendo 
# -> state=({1,3,5}, {0,2,4,6,7}) prendo quindi il secondo array di item dei sets (il secondo set), il quarto e il sesto (tenere conto indice array)

In [8]:
# L'obbiettivo è che tutti gli elementi devo essere presi con un certo numero di set dello stato. La qualità di una soluzione (lo stato scelto) è dato dal minore numero di set presi per prendere tutti gli elementi
# bisogna fare un or tra tutti gli elementi di tutti i set dello stato per vedere se tra tutti i set un certo elemento c'è oppure no (true è presente, false no)
# con all si controlla se ci sono tutti gli elementi cioè se la reduce mi restituisce un array di true che vuol dire che ci sono tutti gli elementi tra i set presi
def goal_check(state):
    return np.all(reduce(np.logical_or, [SETS[i] for i in state.taken], np.array([False for _ in range(PROBLEM_SIZE)])))


# calcolare la distanza dallo stato al goal finale (per una ricerca greedy), in poche parole ritorna quanti elementi che devono essere presi mancano
def distance(state):
    return PROBLEM_SIZE - sum(
        reduce(np.logical_or, [SETS[i] for i in state.taken], np.array([False for _ in range(PROBLEM_SIZE)])))

In [ ]:
assert goal_check(State(set(range(NUM_SETS)), set())), "Problem not solvable"

In [ ]:
# implementazione usando algoritmo di ricerca in ampiezza (Breadth first) usando una Simple Queue che sarebbe una FIFO
# se usassimo una LifoQueue diventerebbe una ricerca in profondità (Depth first)

frontier = SimpleQueue()
frontier.put(State(set(), set(range(NUM_SETS))))

counter = 0
current_state = frontier.get()
while not goal_check(current_state):
    counter += 1
    for action in current_state[1]:
        new_state = State(current_state.taken ^ {action},
                          current_state.not_taken ^ {action})  # {1,2,3} ^ {2} = {1,3}  {1,3} ^ {2} = {1,2,3}
        frontier.put(new_state)
    current_state = frontier.get()

print(f"Solved in {counter:,} steps")
current_state

In [ ]:
# implementazione algoritmo di ricerca usando Dijkstra
# il costo usato per la PriorQueue è il numero di elementi in un set cioè quelli taken

frontier = PriorityQueue()
frontier.put((0, (State(set(), set(range(NUM_SETS))))))

counter = 0
current_state = frontier.get()
while not goal_check(current_state[1]):
    counter += 1
    for action in current_state[1][1]:
        new_state = (current_state[0] + 1, State(current_state[1].taken ^ {action}, current_state[1].not_taken ^ {
            action}))  # {1,2,3} ^ {2} = {1,3}  {1,3} ^ {2} = {1,2,3}
        frontier.put(new_state)
    current_state = frontier.get()

print(f"Solved in {counter:,} steps")
current_state

In [ ]:
# implementazione algoritmo di ricerca greedy

# frontier = PriorityQueue()
frontier = SimpleQueue()
state = State(set(), set(range(NUM_SETS)))
frontier.put((distance(state), state))

counter = 0
_, current_state = frontier.get()
while not goal_check(current_state):
    counter += 1
    for action in current_state[1]:
        new_state = State(current_state.taken ^ {action},
                          current_state.not_taken ^ {action})  # {1,2,3} ^ {2} = {1,3}  {1,3} ^ {2} = {1,2,3}
        frontier.put((distance(state), state))
    _, current_state = frontier.get()

print(f"Solved in {counter:,} steps")
current_state

In [ ]:
#CODICE DI TEST

frontier = PriorityQueue()
frontier.put((0, (State(set(), set(range(NUM_SETS))))))

current_state = frontier.get()
current_state